# Pipeline de Produção — Selagem FB17

Notebook orquestrador agendável como Seeq job.  
Executa a cada 1 hora: extrai dados, atualiza datasets analíticos, avalia gatilhos e grava no SharePoint.

**Fluxo**
1. Generators: `sku_dates` (Seeq) + `00_hour_prev` → `01_vida_rul` → `02_sinais_forca` → `03_padroes`
2. SKU normalization: adiciona `Media_norm` ao sinal de força
3. Avaliação de gatilhos v4.0: RISCO / CRÍTICO (AVISO / CONFIRMADO / FIM_DE_VIDA)
4. Gravação SharePoint (`ESCREVER_SP = True`)

In [1]:
import sys
from pathlib import Path

# notebooks/ para mock seeq em dev local; lplcc_selagem/ para imports src.*
_NB_DIR = Path.cwd()
_ROOT   = _NB_DIR.parent
for _p in (_NB_DIR, _ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from seeq import spy
spy.jobs.schedule("every 1 hour")

,Schedule,Scheduled,Next Run
0,every 1 hour,Every hour,2026-06-03 17:00:00 BRT


,Schedule,Scheduled,Next Run
0,every 1 hour,Every hour,2026-06-03 17:00:00 BRT


In [2]:
# ── Configuração ──────────────────────────────────────────────────────────────
MAQUINA     = "FB17"
LIST_NAME   = "Gatilhos_Selagem"
SP_URL      = ""    # defina SP_URL=https://... em sharepoint.ev
ESCREVER_SP = True  # alterar para True em produção

PATH_HOUR_PREV = _NB_DIR / "00_hour_prev.csv"
PATH_VIDA_RUL  = _NB_DIR / "01_vida_rul.csv"
PATH_WEIBULL   = _NB_DIR / "01_weibull_params.json"
PATH_SINAIS    = _NB_DIR / "02_sinais_forca.csv"
PATH_PADROES   = _NB_DIR / "03_padroes.csv"
PATH_SKU_DATES = _NB_DIR / "sku_dates.csv"
PATH_TROCA     = "troca_modulo.csv"
PATH_STATE     = _NB_DIR / "state_fb17.json"

In [ ]:
# ── Sync troca_modulo.csv ← SharePoint (CONSUMO MAINTACKER BCM.xlsx) ─────────
from src.sp_troca_sync import sync_troca_modulo

try:
    sync_troca_modulo(config_path=_ROOT / "config.yaml", out_path=PATH_TROCA)
except Exception as _e_sync:
    print(f"[sp_troca_sync] AVISO: falha ao sincronizar com SP — usando CSV local. ({_e_sync})")


## Etapa 1 — Generators

Atualiza os quatro datasets analíticos a partir dos dados mais recentes do Seeq.

In [3]:
from src.generators.gen_sku_dates  import run as gen_sku_dates
from src.generators.gen_hour_prev  import run as gen_hour_prev
from src.generators.gen_vida_rul   import run as gen_vida_rul
from src.generators.gen_sinais     import run as gen_sinais
from src.generators.gen_padroes    import run as gen_padroes

print("[0/4] Atualizando Phantom dates via Seeq...")
df_sku_raw = gen_sku_dates(output_path=PATH_SKU_DATES)
n_phantom = df_sku_raw["phantom"].notna().sum()
print(f"      sku_dates: {len(df_sku_raw):,} leituras | {n_phantom:,} com Phantom Code")

print("[1/4] Extraindo dados do Seeq...")
df_hour = gen_hour_prev(output_path=PATH_HOUR_PREV, troca_csv=PATH_TROCA)
print(f"      00_hour_prev: {len(df_hour):,} leituras")

print("[2/4] Calculando vida e Weibull...")
df_vida = gen_vida_rul(
    input_path=PATH_HOUR_PREV,
    output_csv=PATH_VIDA_RUL,
    output_json=PATH_WEIBULL,
    troca_csv=PATH_TROCA,
)
n_ciclos = df_vida["ciclo_id"].nunique() if "ciclo_id" in df_vida.columns else "?"
print(f"      01_vida_rul: {len(df_vida):,} linhas | {n_ciclos} ciclos")

print("[3/4] Calculando features de força...")
df_sinais = gen_sinais(
    input_path=PATH_HOUR_PREV,
    output_path=PATH_SINAIS,
    troca_csv=PATH_TROCA,
)
print(f"      02_sinais_forca: {len(df_sinais):,} linhas")

print("[4/4] Calculando padrões de detecção...")
df_padroes = gen_padroes(
    input_vida_rul=PATH_VIDA_RUL,
    input_sinais=PATH_SINAIS,
    output_path=PATH_PADROES,
    troca_csv=PATH_TROCA,
)
n_gen = df_padroes["genuino"].sum() if "genuino" in df_padroes.columns else "?"
print(f"      03_padroes: {len(df_padroes)} ciclos ({n_gen} genuínos)")

,ID,Type,Time,Count,Pages,Data Processed,Result
0,D3DB6716-C46D-4A54-8990-C32ECEE9E5FE,Signal,00:00:00.85,3449,1,55 KB,Success
1,55D2EF3F-D680-4D70-9F36-04F1A9584FB4,Signal,00:00:01.02,3536,1,56 KB,Success


/home/datalab/fb17-datalab/src/predictor.py:39: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dates = pd.to_datetime(df[col], utc=True).sort_values().tolist()


      00_hour_prev: 3,190 leituras
[2/4] Calculando vida e Weibull...
      01_vida_rul: 3,190 linhas | 42 ciclos
[3/4] Calculando features de força...


/home/datalab/fb17-datalab/src/predictor.py:39: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dates = pd.to_datetime(df[col], utc=True).sort_values().tolist()
/home/datalab/fb17-datalab/src/predictor.py:39: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dates = pd.to_datetime(df[col], utc=True).sort_values().tolist()


      02_sinais_forca: 3,190 linhas
[4/4] Calculando padrões de detecção...
      03_padroes: 40 ciclos (35 genuínos)


/home/datalab/fb17-datalab/src/predictor.py:39: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dates = pd.to_datetime(df[col], utc=True).sort_values().tolist()


## Etapa 2 — Normalização por Phantom

Carrega `02_sinais_forca.csv` já com `Media_norm` gerada por phantom auto-calibrado.  
Elimina o Paradoxo de Simpson sem catálogo estático — fator calibrado pelos primeiros 10 dias de cada ciclo.

In [4]:
import pandas as pd

# 02_sinais_forca.csv já tem Media_norm gerada por gen_sinais (phantom auto-calibrado)
df_hourly = pd.read_csv(PATH_SINAIS, parse_dates=["Timestamp"])
df_hourly["Timestamp"] = pd.to_datetime(df_hourly["Timestamp"], utc=True)
df_hourly = df_hourly.set_index("Timestamp").sort_index()

n_norm    = df_hourly["Media_norm"].notna().sum() if "Media_norm" in df_hourly.columns else 0
n_total   = len(df_hourly)
cobertura = n_norm / n_total if n_total > 0 else 0.0
print(f"Media_norm: {n_norm:,}/{n_total:,} leituras ({cobertura:.0%})")

if n_norm > 0 and "phantom_fator" in df_hourly.columns:
    fator_medio  = df_hourly["phantom_fator"].mean()
    moda_phantom = df_hourly["phantom_codigo"].mode()
    phantom_dom  = moda_phantom.iloc[0] if not moda_phantom.empty else None
    print(f"Fator phantom médio : {fator_medio:.3f}")
    if phantom_dom is not None:
        print(f"Phantom dominante   : {phantom_dom}")

# Usa Media_norm se cobertura >= 50%; caso contrário, fallback para força bruta
media_col = "Media_norm" if cobertura >= 0.50 else "Media"
print(f"Coluna p/ trigger: '{media_col}'")

Media_norm: 3,190/3,190 leituras (100%)
Fator phantom médio : 0.998
Phantom dominante   : 43033322.0
Coluna p/ trigger: 'Media_norm'


## Etapa 3 — Estado do ciclo atual

In [5]:
from src.predictor import load_troca_dates

troca_dates  = load_troca_dates(PATH_TROCA)
ultima_troca = troca_dates[-1]
hoje         = df_hourly.index[-1].normalize()
idade_dias   = (hoje - ultima_troca).days

print(f"Última troca confirmada : {ultima_troca.date()}")
print(f"Última leitura          : {hoje.date()}")
print(f"Idade do rolo           : {idade_dias} dias")

Última troca confirmada : 2026-05-16
Última leitura          : 2026-06-02
Idade do rolo           : 17 dias


/home/datalab/fb17-datalab/src/predictor.py:39: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dates = pd.to_datetime(df[col], utc=True).sort_values().tolist()


## Etapa 4 — Avaliação de gatilhos

In [6]:
import os

sp_client = None
if ESCREVER_SP:
    try:
        from dotenv import dotenv_values
        # sharepoint.ev fica na raiz do projeto (fb17-datalab/), não em notebooks/
        _ev = dotenv_values(_ROOT / "sharepoint.ev")
        _sp_url  = _ev.get("SP_URL") or "https://kimberlyclark.sharepoint.com/Sites/H945"
        _sp_user = _ev.get("SP_USER")
        _sp_pass = _ev.get("SP_PASS")
        from src.sharepoint_methods import SharePointClient
        sp_client = SharePointClient(url=_sp_url, username=_sp_user, password=_sp_pass)
        print(f"SharePoint: conectado ({_sp_url})")
    except Exception as _e:
        print(f"SharePoint: falha na conexão ({_e}) — modo dry-run")
else:
    print("SharePoint: dry-run (ESCREVER_SP=False)")

SharePoint: conectado (https://kimberlyclark.sharepoint.com/Sites/H945)


In [7]:
# ── Contexto de paradas (anomaly_delay) ───────────────────────────────────
import pandas as pd
PATH_DELAY = _NB_DIR / "anomaly_delay.csv"
df_delay = None
try:
    df_delay = pd.read_csv(PATH_DELAY, index_col="Timestamp", parse_dates=True)
    df_delay.index = df_delay.index.tz_convert("UTC")
    print(f"anomaly_delay: {len(df_delay):,} leituras | stopped_h_max={df_delay['stopped_h'].max():.1f}h")
except FileNotFoundError:
    print("anomaly_delay.csv não encontrado — age_risk sem correção de paradas")

anomaly_delay: 1,202 leituras | stopped_h_max=0.0h


In [8]:
from src.trigger_engine import TriggerEngine

engine = TriggerEngine(maquina=MAQUINA, state_path=PATH_STATE)

print(f"Avaliando {hoje.date()} | rolo com {idade_dias} dias | sinal='{media_col}'")
events = engine.evaluate(
    df_hourly=df_hourly,
    troca_date=ultima_troca,
    sp_client=sp_client,
    list_name=LIST_NAME,
    media_col=media_col,
    troca_dates=troca_dates,
    df_delay=df_delay,
)

print()
if events:
    print(f"{len(events)} disparo(s):")
    for ev in events:
        sp_tag = f" [SP ID={ev.sp_item_id}]" if ev.sp_item_id else ""
        print(f"  [{ev.gatilho}] {ev.severidade}{sp_tag}")
        print(f"  {ev.mensagem.splitlines()[0]}")
else:
    print("Nenhum gatilho disparado.")

Avaliando 2026-06-02 | rolo com 17 dias | sinal='Media_norm'
1 items created
1 items created

2 disparo(s):
  [RISCO] INFO [SP ID=162]
  RISCO FB17 — Leitura Anomala Isolada | Evento No1 no ciclo atual | Dia 17 | 570 N
  [CRITICO] ALTA [SP ID=163]
  CRITICO FB17 — Degradacao + Risco Elevado | Evento No2 no ciclo | Dia 17 | 570 N | p_risk=0.40


## Diagnóstico (execução manual)

Snapshot completo dos indicadores calculados para hoje. Útil para debug e auditoria.

In [9]:
from src.trigger_engine import compute_p_risk_snapshot

snap = compute_p_risk_snapshot(df_hourly, ultima_troca, hoje)
print("Indicadores calculados:")
for k, v in snap.items():
    print(f"  {k:<22}: {v}")

Indicadores calculados:
  today                 : 2026-06-02
  age_days              : 17
  mean_3d               : 695.2
  mean_7d               : 748.9
  mean_14d              : 901.6
  min_3d                : 570.3
  mediana_3d            : 703.0
  n_leituras_abaixo_800 : 12
  ratio_3_14            : 0.7711
  slope_7d              : -26.8
  slope_min_ciclo       : -80.6
  proj_48h              : 641.6
  age_risk              : 0.2252
  sig_score             : 0.5373
  p_risk                : 0.4958
  cond_aviso            : True
  cond_confirmado_red   : True
  cond_confirmado_forca : True
  cond_risco            : False


In [10]:
# ── Histórico de gatilhos por ciclo ────────────────────────────────────────
#   0 = ciclo atual (aberto)
#  -1 = último ciclo completo
#  -2 = penúltimo  ...
CICLO_OFFSET = -2

import tempfile, json
from pathlib import Path
from src.trigger_engine import TriggerEngine, compute_p_risk_snapshot

# ── Resolver limites do ciclo ─────────────────────────────────────────────
_n = len(troca_dates)
_min_offset = -(_n - 1)   # ex: 15 trocas → offset mínimo = -14

if CICLO_OFFSET > 0 or CICLO_OFFSET < _min_offset:
    raise ValueError(
        f"CICLO_OFFSET deve estar entre {_min_offset} e 0 "
        f"(há {_n - 1} ciclos completos + ciclo atual)."
    )

if CICLO_OFFSET == 0:
    t_ini  = troca_dates[-1]
    t_fim  = None
    rotulo = f"atual (desde {t_ini.date()})"
else:
    t_ini  = troca_dates[CICLO_OFFSET - 1]
    t_fim  = troca_dates[CICLO_OFFSET]
    _dur   = (t_fim - t_ini).days
    rotulo = f"ciclo {CICLO_OFFSET:+d}  |  {t_ini.date()} → {t_fim.date()}  ({_dur} dias)"

print(f"━━ Ciclo selecionado: {rotulo} ━━\n")

# ── Filtrar séries do ciclo ───────────────────────────────────────────────
_df_c = df_hourly[df_hourly.index >= t_ini]
if t_fim:
    _df_c = _df_c[_df_c.index < t_fim]

if _df_c.empty:
    print("  Sem dados para este ciclo.")
else:
    # Mini-backtest: reconstruir eventos disparados no ciclo
    _state_tmp = Path(tempfile.mktemp(suffix=".json"))
    _eng       = TriggerEngine(MAQUINA, _state_tmp)
    _dias      = pd.DatetimeIndex(sorted({ts.normalize() for ts in _df_c.index}))

    _eventos = []   # lista de (day, TriggerEvent)
    for _day in _dias:
        _df_ate = df_hourly[df_hourly.index <= _day + pd.Timedelta(hours=23, minutes=59)]
        try:
            for _ev in _eng.evaluate(
                _df_ate,
                troca_date=t_ini,
                sp_client=None,
                today=_day,
                troca_dates=troca_dates,
                media_col=media_col,
                df_delay=df_delay,
            ):
                _eventos.append((_day, _ev))
        except Exception as _exc:
            print(f"  [ERRO] {_day.date()}: {_exc}")

    if _state_tmp.exists():
        _state_tmp.unlink()

    # ── Sumário ─────────────────────────────────────────────────────────
    if not _eventos:
        print("  Nenhum gatilho disparado neste ciclo.")
    else:
        print(f"  {len(_eventos)} disparo(s):\n")
        for _d, _ev in _eventos:
            _mark = "  ◀ último" if (_d, _ev) is _eventos[-1] else ""
            print(f"    {_d.date()}  [{_ev.severidade:<6}]  {_ev.gatilho}{_mark}")

        # ── Detalhe do último disparo ─────────────────────────────────
        _day_ult, _ev_ult = _eventos[-1]
        print(f"\n── Último disparo: {_ev_ult.gatilho}  "
              f"em {_day_ult.date()}  (dia {_ev_ult.idade_maintacker} do ciclo) ──\n")

        _snap = compute_p_risk_snapshot(df_hourly, t_ini, _day_ult)
        for _k, _v in _snap.items():
            print(f"  {_k:<22}: {_v}")

        print("\n── Payload enviado (TeamsPayload) ──")
        if _ev_ult.teams_payload:
            try:
                print(json.dumps(json.loads(_ev_ult.teams_payload), indent=2, ensure_ascii=False))
            except Exception:
                print(_ev_ult.teams_payload)
        else:
            print("  (sem payload — dry-run)")

[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] CRITICO(CONFIRMADO) nao persistido (sp_client=None).
[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] CRITICO(CONFIRMADO) nao persistido (sp_client=None).
[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] CRITICO(CONFIRMADO) nao persistido (sp_client=None).
[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] CRITICO(AVISO) nao persistido (sp_client=None).
[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] CRITICO(CONFIRMADO) nao persistido (sp_client=None).
[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] CRITICO(CONFIRMADO) nao persistido (sp_client=None).
[dry-run] RISCO(None) nao persistido (sp_client=None).
[dry-run] CRITICO(CONFIRMADO) nao persistido (sp_client=None).


━━ Ciclo selecionado: ciclo -2  |  2026-03-21 → 2026-04-18  (28 dias) ━━

  17 disparo(s):

    2026-03-22  [INFO  ]  RISCO
    2026-03-24  [INFO  ]  RISCO
    2026-03-26  [INFO  ]  RISCO
    2026-04-03  [INFO  ]  RISCO
    2026-04-04  [ALTA  ]  CRITICO
    2026-04-05  [INFO  ]  RISCO
    2026-04-06  [ALTA  ]  CRITICO
    2026-04-07  [INFO  ]  RISCO
    2026-04-08  [ALTA  ]  CRITICO
    2026-04-09  [INFO  ]  RISCO
    2026-04-10  [MEDIA ]  CRITICO
    2026-04-12  [INFO  ]  RISCO
    2026-04-12  [ALTA  ]  CRITICO
    2026-04-14  [INFO  ]  RISCO
    2026-04-14  [ALTA  ]  CRITICO
    2026-04-16  [INFO  ]  RISCO
    2026-04-16  [ALTA  ]  CRITICO

── Último disparo: CRITICO  em 2026-04-16  (dia 26 do ciclo) ──

  today                 : 2026-04-16
  age_days              : 26
  mean_3d               : 1249.2
  mean_7d               : 1047.4
  mean_14d              : 985.6
  min_3d                : 667.4
  mediana_3d            : 1304.8
  n_leituras_abaixo_800 : 1
  ratio_3_14            : 1

In [17]:
from seeq import spy
import pandas as pd

batch_size = 1000
max_rows = 10000
all_results = []
page = 0

while True:
    df = spy.search(
        {
            'Name': '*brsz*',
            'Type': 'Signal',
            # 'Datasource Class': 'OSIsoft PI',  # Descomente se quiser só PI System
        },
        all_properties=True,
        old_asset_format=False,
        limit=batch_size,
        order_by='ID'
    )
    if df.empty:
        break

    all_results.append(df)
    page += 1
    print(f'Página {page}: {len(df)} itens (até ID {df["ID"].iloc[-1]})')

    # Checa se já atingiu o máximo desejado
    if sum(len(x) for x in all_results) >= max_rows:
        print(f'Limite de {max_rows} atingido.')
        break

    # Se retornaram menos que batch_size, acabou!
    if len(df) < batch_size:
        break

# Junta todos os resultados e limita a 10000 linhas
search_df = pd.concat(all_results, ignore_index=True).head(max_rows)
print(f'Total de sinais encontrados: {len(search_df)}')
search_df.head()


,Name,Type,Time,Count,Pages,Result
0,*brsz*,Signal,00:00:03.28,2000,2,Querying


Página 10: 1000 itens (até ID 0184ADA0-1260-4DB0-ACBB-80DEC99C0587)
Limite de 10000 atingido.
Total de sinais encontrados: 10000


,ID,Path,Asset,Name,Description,Type,Value Unit Of Measure,Datasource Name,Archived,Source Value Unit Of Measure,...,Compressing,Display Units,Instrument Tag,is_crap,dead_signal_reason,partition_key,search_payload,search_term,search_request_index,Extended Descriptor
0,0000AEF9-CB10-4C6A-A8B0-9C90F0A32106,NaN,NaN,BRSZ PC02 Quality - PC02_2012_A1_PROD_Cint:Elá...,Plant Apps VarID: 17692,StoredSignal,,BRSZAS70 PA,False,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0000C934-8A4E-4C40-8485-8A523649703D,_LAO >> Suzano,Plant Apps,BRSZ AC22 Quality - AC22_Bag1_BPP1_UoM,Plant Apps VarID: 40375,CalculatedSignal,,\trees\LAO,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00013CEB-CE7D-406A-B607-87B196B9FC94,_LAO >> Suzano,Plant Apps,BRSZ PC01 Quality - PC01_REI-Fralda_Fraldas/Em...,Plant Apps VarID: 39585,CalculatedSignal,,\trees\LAO,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000141FA-A397-435A-B1AA-F4A966A99D8A,NaN,NaN,BRSZ TRN Quality - TRN_2014_A1_PROD_OrelhaT:V14-I,Plant Apps VarID: 33422,StoredSignal,,BRSZAS70 PA SQL,False,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0001E492-50AA-4648-B435-228DC1860056,_LAO >> Suzano,Plant Apps,BRSZ AF17 Quality - AF17_5013_A2_PALLET_Emb:V1...,Plant Apps VarID: 2649,CalculatedSignal,,\trees\LAO,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
search_df

,ID,Path,Asset,Name,Description,Type,Value Unit Of Measure,Datasource Name,Archived,Source Value Unit Of Measure,...,Compressing,Display Units,Instrument Tag,is_crap,dead_signal_reason,partition_key,search_payload,search_term,search_request_index,Extended Descriptor
0,0000AEF9-CB10-4C6A-A8B0-9C90F0A32106,NaN,NaN,BRSZ PC02 Quality - PC02_2012_A1_PROD_Cint:Elá...,Plant Apps VarID: 17692,StoredSignal,,BRSZAS70 PA,False,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0000C934-8A4E-4C40-8485-8A523649703D,_LAO >> Suzano,Plant Apps,BRSZ AC22 Quality - AC22_Bag1_BPP1_UoM,Plant Apps VarID: 40375,CalculatedSignal,,\trees\LAO,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00013CEB-CE7D-406A-B607-87B196B9FC94,_LAO >> Suzano,Plant Apps,BRSZ PC01 Quality - PC01_REI-Fralda_Fraldas/Em...,Plant Apps VarID: 39585,CalculatedSignal,,\trees\LAO,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000141FA-A397-435A-B1AA-F4A966A99D8A,NaN,NaN,BRSZ TRN Quality - TRN_2014_A1_PROD_OrelhaT:V14-I,Plant Apps VarID: 33422,StoredSignal,,BRSZAS70 PA SQL,False,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0001E492-50AA-4648-B435-228DC1860056,_LAO >> Suzano,Plant Apps,BRSZ AF17 Quality - AF17_5013_A2_PALLET_Emb:V1...,Plant Apps VarID: 2649,CalculatedSignal,,\trees\LAO,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0183AF6A-EC07-4AB9-8E93-A37BA7A68108,NaN,NaN,BRSZ AC22 Quality - AC22_9902_B1_Larg CD extre...,Plant Apps VarID: 61904,StoredSignal,,BRSZAS70 PA,False,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9996,0183E462-19C7-4C45-9176-3C40ECFD231C,NaN,NaN,BRSZ FB13 Quality - FB13_7383_A1_Altura da Bar...,Plant Apps VarID: 10414,StoredSignal,,BRSZAS70 PA,False,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9997,0184843D-F6B4-4E09-B5E3-6D8C1CFEC990,NaN,NaN,BRSZ TRN Quality - TRN_2008_A1_PROD_Cintura:V8-Fr,Plant Apps VarID: 33364,StoredSignal,,BRSZAS70 PA,False,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9998,0184A7A8-10FA-48C4-968F-71BB996F06E6,NaN,NaN,BRSZ FB16 Quality - FB16_1010_A1_PROD_V10-Furo...,Plant Apps VarID: 8962,StoredSignal,,BRSZAS70 PA SQL,False,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
